In [ ]:
import torch
print(torch.cuda.is_available())  
print(torch.cuda.get_device_name(0))  


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import cv2
import os
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from sklearn.metrics import jaccard_score
import glob
import onnx
import tensorflow as tf
from tensorflow import keras

# Custom dataset class for loading sea turtle images and masks
class SeaTurtleDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        # Filter out image and mask paths to include only certain file formats
        self.image_paths = sorted([p for p in image_paths if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.mask_paths = sorted([p for p in mask_paths if p.lower().endswith('.png') and '_channel_' in p and not p.endswith('_channel_0.png')])
        self.transform = A.Compose([A.Resize(256, 256), *transform.transforms]) if transform else A.Compose([A.Resize(256, 256)])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load an image and its corresponding mask
        image = cv2.imread(self.image_paths[idx], cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Failed to load image at path: {self.image_paths[idx]}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_number = os.path.basename(self.image_paths[idx]).split('_')[0]
        mask_paths = [p for p in self.mask_paths if os.path.basename(p).split('_')[0] == image_number and not p.endswith('_channel_0.png')]
        if len(mask_paths) == 0:
            raise ValueError(f"No matching masks found for image with number: {image_number}")
        mask_paths = sorted(mask_paths, key=lambda x: int(x.split('_channel_')[-1].split('.')[0]))
        print(f"Loading mask paths for image number {image_number}: {mask_paths}")
        mask = self.load_multichannel_mask(mask_paths)
        
        # Apply augmentations if provided
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask

    def load_multichannel_mask(self, mask_paths):
        # Load individual channel masks and combine them into a multi-class mask
        first_mask = cv2.imread(mask_paths[0], cv2.IMREAD_GRAYSCALE)
        if first_mask is None:
            raise ValueError(f"Failed to load the first channel mask at path: {mask_paths[0]}")
        mask = np.zeros(first_mask.shape, dtype=np.uint8)
        for i, mask_path in enumerate(mask_paths):
            channel_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if channel_mask is None:
                raise ValueError(f"Failed to load channel mask at path: {mask_path}")
            mask[channel_mask > 0] = i + 1
        return mask

# Data augmentation and transformation
transform = A.Compose([
    A.HorizontalFlip(p=0.5),  # Horizontal flip with 50% probability
    A.RandomRotate90(p=0.5),  # Randomly rotate the image by 90 degrees with 50% probability
    A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=15, shift_limit=0.1, p=0.5),  # Random shift, scale, and rotate
    A.RandomBrightnessContrast(p=0.5),  # Random brightness and contrast adjustment
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # Normalization to match pre-trained models
    ToTensorV2()  # Convert to PyTorch tensor
])

# Load the dataset
print("Loading datasets...")
valid_image_paths = [os.path.join(root, file) for root, _, files in os.walk('./preprocessed_data/valid/images') for file in files if file.lower().endswith(('.jpg', '.jpeg', '.png'))]
valid_mask_paths = [os.path.join(root, file) for root, _, files in os.walk('./preprocessed_data/valid/masks') for file in files if file.lower().endswith('.png')]
valid_dataset = SeaTurtleDataset(valid_image_paths, valid_mask_paths, transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)
train_image_paths = [os.path.join(root, file) for root, _, files in os.walk('./preprocessed_data/train/images') for file in files if file.lower().endswith(('.jpg', '.jpeg', '.png'))]
train_mask_paths = [os.path.join(root, file) for root, _, files in os.walk('./preprocessed_data/train/masks') for file in files if file.lower().endswith('.png')]
train_dataset = SeaTurtleDataset(train_image_paths, train_mask_paths, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print("Datasets loaded successfully.")

# Check if the dataset is successfully loaded
print("Checking dataset samples...")
plt.figure(figsize=(20, 20))
for i in range(10):
    sample_image, sample_mask = train_dataset[i]
    plt.subplot(10, 2, 2 * i + 1)
    plt.imshow(sample_image.permute(1, 2, 0))
    plt.title(f"Sample Image {i + 1}")
    plt.axis('off')
    plt.subplot(10, 2, 2 * i + 2)
    plt.imshow(sample_mask, cmap='tab20')
    plt.title(f"Sample Mask {i + 1}")
    plt.axis('off')
plt.tight_layout()
plt.show()
print("Dataset samples checked.")

# Define the custom UNet model
print("Initializing custom UNet model...")

class UNet(nn.Module):
    def __init__(self, num_classes):
        super(UNet, self).__init__()
        self.encoder1 = self.block(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.encoder2 = self.block(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.encoder3 = self.block(128, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder1 = self.block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder2 = self.block(128, 64)

        self.output_layer = nn.Conv2d(64, num_classes, kernel_size=1)

    def block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        x = self.pool1(enc1)

        enc2 = self.encoder2(x)
        x = self.pool2(enc2)

        x = self.encoder3(x)

        x = self.up1(x)
        x = torch.cat([x, enc2], dim=1)
        x = self.decoder1(x)

        x = self.up2(x)
        x = torch.cat([x, enc1], dim=1)
        x = self.decoder2(x)

        return self.output_layer(x)

model = UNet(num_classes=4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Custom UNet model initialized.")

# Define the loss function, optimizer, and learning rate scheduler
loss_fn = nn.CrossEntropyLoss(ignore_index=0)  # Cross-entropy loss for multi-class segmentation, ignoring background
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Adam optimizer with a learning rate of 0.001
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)  # Reduce learning rate if validation loss plateaus

# Train the model
epochs = 30
early_stopping_patience = 5
best_loss = float('inf')
early_stopping_counter = 0

print("Starting training...")
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    model.train()
    epoch_loss = 0
    epoch_jaccard = np.zeros(3)
    for batch_idx, (images, masks) in enumerate(train_loader):
        print(f"Training batch {batch_idx + 1}/{len(train_loader)}", flush=True)
        images = images.to(device)
        masks = masks.long().to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Calculate Jaccard Index (IoU) for the batch
        preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
        masks_np = masks.cpu().numpy()
        for i in range(1, 4):
            if np.any(masks_np == i):
                jaccard = jaccard_score((masks_np == i).flatten(), (preds == i).flatten(), average='binary')
                epoch_jaccard[i - 1] += jaccard

        # Output example images
        if batch_idx % 10 == 0:
            plt.figure(figsize=(24, 12))
            plt.subplot(2, 4, 1)
            plt.imshow(images[0].permute(1, 2, 0).cpu().numpy())
            plt.title("Input Image")
            plt.axis('off')
            plt.subplot(2, 4, 2)
            plt.imshow(preds[0] == 1, cmap='gray', vmin=0, vmax=1)
            plt.title("Predicted Shell (Class 1)")
            plt.axis('off')
            plt.subplot(2, 4, 3)
            plt.imshow(preds[0] == 2, cmap='gray', vmin=0, vmax=1)
            plt.title("Predicted Limbs (Class 2)")
            plt.axis('off')
            plt.subplot(2, 4, 4)
            plt.imshow(preds[0] == 3, cmap='gray', vmin=0, vmax=1)
            plt.title("Predicted Head (Class 3)")
            plt.axis('off')
            plt.subplot(2, 4, 5)
            plt.imshow(masks_np[0] == 1, cmap='gray', vmin=0, vmax=1)
            plt.title("True Shell (Class 1)")
            plt.axis('off')
            plt.subplot(2, 4, 6)
            plt.imshow(masks_np[0] == 2, cmap='gray', vmin=0, vmax=1)
            plt.title("True Limbs (Class 2)")
            plt.axis('off')
            plt.subplot(2, 4, 7)
            plt.imshow(masks_np[0] == 3, cmap='gray', vmin=0, vmax=1)
            plt.title("True Head (Class 3)")
            plt.axis('off')
            plt.tight_layout()
            plt.show()

    epoch_loss /= len(train_loader)
    epoch_jaccard /= len(train_loader)

    print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {epoch_loss}, Training Jaccard Index: {epoch_jaccard.tolist()}", flush=True)

    # Validate the model
    print("Validating model...")
    model.eval()
    valid_loss = 0
    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(valid_loader):
            print(f"Validation batch {batch_idx + 1}/{len(valid_loader)}", flush=True)
            images = images.to(device)
            masks = masks.long().to(device)

            outputs = model(images)
            loss = loss_fn(outputs, masks)
            valid_loss += loss.item()

            # Output example images
            if batch_idx % 10 == 0:
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                plt.figure(figsize=(12, 6))
                plt.subplot(1, 2, 1)
                plt.imshow(images[0].permute(1, 2, 0).cpu().numpy())
                plt.title("Validation Input Image")
                plt.axis('off')
                plt.subplot(1, 2, 2)
                plt.imshow(np.where((preds[0] > 0) & (preds[0] <= 3), preds[0], 0), cmap='tab20')
                plt.title("Validation Predicted Mask")
                plt.axis('off')
                plt.show()

    valid_loss /= len(valid_loader)
    print(f'Epoch {epoch + 1}/{epochs}, Training Loss: {epoch_loss}, Validation Loss: {valid_loss}', flush=True)

    # Adjust the learning rate
    scheduler.step(valid_loss)

    # Early stopping
    if valid_loss < best_loss:
        best_loss = valid_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), 'best_unet_seaturtle.pth')
        print("Best model saved.", flush=True)
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_patience:
            print("Early stopping triggered.", flush=True)
            break

print("Training completed.")



In [ ]:
# Convert the PyTorch model to ONNX format
print("Converting model to ONNX format...")
dummy_input = torch.randn(1, 3, 256, 256).to(device)
torch.onnx.export(model, dummy_input, "v2_unet_seaturtle.onnx", opset_version=11)
print("Model converted to ONNX format.")


In [ ]:
# Calculate and print IoU and mIoU for each class
print("Calculating IoU and mIoU for each class...")
model.eval()
iou_per_class = np.zeros(3)
num_batches = len(valid_loader)
with torch.no_grad():
    for batch_idx, (images, masks) in enumerate(valid_loader):
        images = images.to(device)
        masks = masks.long().to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        masks_np = masks.cpu().numpy()

        for i in range(1, 4):
            if np.any(masks_np == i):
                iou = jaccard_score((masks_np == i).flatten(), (preds == i).flatten(), average='binary')
                iou_per_class[i - 1] += iou

# Calculate average IoU for each class
iou_per_class /= num_batches
for i, iou in enumerate(iou_per_class, 1):
    print(f"IoU for Class {i}: {iou:.4f}")

# Calculate and print mIoU
miou = np.mean(iou_per_class)
print(f"Mean IoU (mIoU): {miou:.4f}")